In [ ]:
import torch
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error, mean_squared_error, r2_score
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, MSELoss
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam

In [70]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        self._y = torch.from_numpy(labels).float()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        int(idx)
        return self._x[idx], self._y[idx]

In [71]:
def train_model(features, targets, model, kf, batch, epochs, optimizer, criterion):
    r2_scores = []
    rmse_scores = []
    mse_scores = []
    for i, (train_index, test_index) in enumerate(kf.split(features)):
        print(f"treinando model -------- fold {i + 1}")
        train_data = MyDataset(features[train_index], targets[train_index])
        train_loader = DataLoader(train_data, batch_size = batch)
        for i_epochs in range(epochs):
            model.train()
            for x, y in train_loader:
                optimizer.zero_grad()
                output = model(x)
                loss =  criterion(output, y) #calcular meu erro com mey y verdadeiro
                loss.backward()
                optimizer.step()

        model.eval()
        with torch.no_grad():
            x_test_tensor = Tensor(features[test_index]).float()
            y_true = targets[test_index]
            result = model(x_test_tensor).cpu().numpy()

            r2_s = r2_score(y_true, result)
            rmse_s = root_mean_squared_error(y_true, result)
            mse_s = mean_squared_error(y_true, result)

            r2_scores.append(r2_s)
            rmse_scores.append(rmse_s)
            mse_scores.append(mse_s)

            print(f"R^2: {r2_s}")
            print(f"RMSE: {rmse_s}")
            print(f"MSE: {mse_s}")

    print(f"\nResultado modelo ----- R^2: {np.mean(r2_scores)}, RMSE: {np.mean(rmse_scores)}, MSE: {np.mean(mse_scores)}")


In [72]:
dataset = load_diabetes()
features = dataset.data
targets = np.expand_dims(dataset.target, axis=1)

#print(targets.shape)

In [73]:
model = Sequential(
    Linear(10,16),
    ReLU(),
    Linear(16,16),
    ReLU(),
    Linear(16,1)
)

In [74]:
optimizer = Adam(model.parameters(), lr=0.001)
criterion = MSELoss()
max_epochs = 100
batch = 12
kf = KFold(n_splits = 5, shuffle=True)

In [75]:
train_model(features, targets, model, kf, batch, max_epochs, optimizer, criterion)


treinando model -------- fold 1
R^2: 0.3572096337869718
RMSE: 61.11311341174865
MSE: 3734.8126308772526
treinando model -------- fold 2
R^2: 0.4223019468712912
RMSE: 51.30759787890671
MSE: 2632.4696001035923
treinando model -------- fold 3
R^2: 0.5386824387756857
RMSE: 56.01230453916135
MSE: 3137.3782597877553
treinando model -------- fold 4
R^2: 0.43546684123397283
RMSE: 56.6680104097295
MSE: 3211.2634037972107
treinando model -------- fold 5
R^2: 0.6070457394763098
RMSE: 51.01230394586648
MSE: 2602.2551538654643

Resultado modelo ----- R^2: 0.47214132002884635, RMSE: 55.22266603708253, MSE: 3063.6358096862555
